# 📦 Notebook 01 — Data Loading and Cleaning

**Purpose:** Load all 85 Kickstarter CSV shards, deduplicate, filter to binary outcomes, perform a temporal train/test split, parse JSON columns, remove leakage and junk columns, audit missing values, and save clean parquet files for downstream notebooks.

**Outputs saved to `../outputs/`:**
- `clean_df.parquet` — full cleaned dataset
- `train_df.parquet` — temporal train split (oldest 80%)
- `test_df.parquet` — temporal test split (newest 20%)
- `leakage_columns.json` — reference list of dropped columns


## Setup — Imports and Paths

In [1]:
import os, sys, glob, json, re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust DATA_PATH if running from a different working directory
DATA_PATH    = "../data/Kickstarter_2026-02-12T03_20_22_018Z"
OUTPUTS_PATH = "../outputs"
os.makedirs(OUTPUTS_PATH, exist_ok=True)

print(f"Data folder : {os.path.abspath(DATA_PATH)}")
print(f"Outputs path: {os.path.abspath(OUTPUTS_PATH)}")


Data folder : /Users/dacobri/Desktop/MSc Business Analytics/Classes Term 2/Artificial Intelligence 2/Final Project/Kickstarter_ML_Project/data/Kickstarter_2026-02-12T03_20_22_018Z
Outputs path: /Users/dacobri/Desktop/MSc Business Analytics/Classes Term 2/Artificial Intelligence 2/Final Project/Kickstarter_ML_Project/outputs


## Step 1 — Load and Merge All 85 CSV Files

We use `glob` to discover all shards, load each with `pd.read_csv(low_memory=False)`,
then concatenate vertically with `ignore_index=True`.  
All 85 files have **identical 42-column schemas** — confirmed in our pre-project analysis.


In [2]:
files = sorted(glob.glob(os.path.join(DATA_PATH, "Kickstarter*.csv")))
print(f"CSV files found: {len(files)}")

parts = []
for f in files:
    try:
        parts.append(pd.read_csv(f, low_memory=False))
    except Exception as e:
        print(f"  [WARN] {os.path.basename(f)}: {e}")

df = pd.concat(parts, ignore_index=True)
print(f"\nShape after concat: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


CSV files found: 85

Shape after concat: (267673, 42)
Columns: ['backers_count', 'blurb', 'category', 'converted_pledged_amount', 'country', 'country_displayable_name', 'created_at', 'creator', 'currency', 'currency_symbol', 'currency_trailing_code', 'current_currency', 'deadline', 'disable_communication', 'fx_rate', 'goal', 'id', 'is_disliked', 'is_in_post_campaign_pledging_phase', 'is_launched', 'is_liked', 'is_starrable', 'launched_at', 'location', 'name', 'percent_funded', 'photo', 'pledged', 'prelaunch_activated', 'profile', 'slug', 'source_url', 'spotlight', 'staff_pick', 'state', 'state_changed_at', 'static_usd_rate', 'urls', 'usd_exchange_rate', 'usd_pledged', 'usd_type', 'video']


## Step 2 — Deduplicate on `id`

The Kickstarter API paginates results, so the same campaign can appear in multiple
shard files. We deduplicate on the unique campaign `id`.

**Expected:** ~58,000 duplicates removed, leaving ~209,000 unique campaigns.


In [3]:
n_before = len(df)
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
n_after = len(df)
print(f"Before deduplication : {n_before:,} rows")
print(f"After deduplication  : {n_after:,} rows")
print(f"Duplicates removed   : {n_before - n_after:,}")


Before deduplication : 267,673 rows
After deduplication  : 209,243 rows
Duplicates removed   : 58,430


## Step 3 — Filter to Binary Outcome

We keep only campaigns with a **definitive outcome**: `successful` or `failed`.  
Other states are dropped:
- `live` — campaign still running, outcome unknown
- `canceled` — creator withdrew; not a success/failure signal
- `suspended` — removed by Kickstarter; atypical cases
- `submitted` — pre-launch draft

**Expected final count:** ~188,000 rows.


In [4]:
print("All state value counts before filtering:")
print(df["state"].value_counts(dropna=False))

dropped_states = df[~df["state"].isin(["successful", "failed"])]["state"].value_counts(dropna=False)
print(f"\nDropped states:")
print(dropped_states)

df = df[df["state"].isin(["successful", "failed"])].reset_index(drop=True)
print(f"\nFinal row count: {len(df):,}")


All state value counts before filtering:
state
successful    117627
failed         70802
canceled        9134
submitted       7192
live            2995
started         1488
suspended          5
Name: count, dtype: int64

Dropped states:
state
canceled     9134
submitted    7192
live         2995
started      1488
suspended       5
Name: count, dtype: int64

Final row count: 188,429


## Step 4 — Create the Binary Target Variable

`success = 1` if the campaign reached its funding goal (`state == "successful"`).  
`success = 0` if it did not (`state == "failed"`).

We check class balance here — an important input to all modelling decisions later.


In [5]:
df["success"] = (df["state"] == "successful").astype(int)

vc = df["success"].value_counts()
print("Class distribution:")
print(f"  success = 1 (funded)  : {vc[1]:>8,}  ({vc[1]/len(df)*100:.1f}%)")
print(f"  success = 0 (failed)  : {vc[0]:>8,}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  Total                 : {len(df):>8,}")
print(f"\nImbalance ratio: 1 : {vc[0]/vc[1]:.2f} (failed:funded)")


Class distribution:
  success = 1 (funded)  :  117,627  (62.4%)
  success = 0 (failed)  :   70,802  (37.6%)
  Total                 :  188,429

Imbalance ratio: 1 : 0.60 (failed:funded)


## Step 5 — Temporal Train/Test Split

### Why Temporal, Not Random?

A **random split** would be the wrong approach here for three reasons:

1. **Future data leakage**: A random split allows the model to train on campaigns from 2023
   and test on campaigns from 2015. The model would effectively see "future" patterns —
   temporal trends, macroeconomic conditions, platform algorithm changes — that would not
   be available in a real deployment setting.

2. **Mimicking real deployment**: In production, this model would be trained on historical
   campaigns and used to predict the success of *new* campaigns. A temporal split
   faithfully replicates this scenario: we train on the oldest 80% and evaluate on
   the most recent 20%.

3. **Distributional shift detection**: By preserving chronological order, we can later
   analyse whether the success rate or feature distributions have shifted over time —
   a critical concern for model longevity.

We sort by `launched_at` (Unix timestamp) and take the first 80% as training data.


In [6]:
# Convert launched_at to numeric (it's sometimes stored as string)
df["launched_at"] = pd.to_numeric(df["launched_at"], errors="coerce")
df = df.sort_values("launched_at").reset_index(drop=True)

# 80/20 chronological split
cutoff_idx = int(len(df) * 0.80)
train_df = df.iloc[:cutoff_idx].copy()
test_df  = df.iloc[cutoff_idx:].copy()

# Human-readable timestamps
cutoff_ts = df["launched_at"].iloc[cutoff_idx]
train_start = pd.to_datetime(df["launched_at"].iloc[0], unit="s", utc=True)
train_end   = pd.to_datetime(df["launched_at"].iloc[cutoff_idx - 1], unit="s", utc=True)
test_start  = pd.to_datetime(df["launched_at"].iloc[cutoff_idx], unit="s", utc=True)
test_end    = pd.to_datetime(df["launched_at"].iloc[-1], unit="s", utc=True)

print(f"Temporal split cutoff: {pd.to_datetime(cutoff_ts, unit='s', utc=True).strftime('%Y-%m-%d')}")
print(f"\nTrain set : {len(train_df):,} rows")
print(f"  Date range: {train_start.strftime('%Y-%m-%d')} → {train_end.strftime('%Y-%m-%d')}")
print(f"  Class balance: {train_df['success'].mean()*100:.1f}% success")
print(f"\nTest set  : {len(test_df):,} rows")
print(f"  Date range: {test_start.strftime('%Y-%m-%d')} → {test_end.strftime('%Y-%m-%d')}")
print(f"  Class balance: {test_df['success'].mean()*100:.1f}% success")


Temporal split cutoff: 2024-06-07

Train set : 150,743 rows
  Date range: 2009-04-25 → 2024-06-07
  Class balance: 60.7% success

Test set  : 37,686 rows
  Date range: 2024-06-07 → 2026-02-06
  Class balance: 69.2% success


## Step 6 — Parse JSON Columns

Two columns contain JSON-encoded metadata: `category` and `location`.  
We extract the most useful fields from each using a safe parser with try/except fallbacks.

- `category` → `cat_name` (e.g. "Tabletop Games"), `cat_parent_name` (e.g. "Games")
- `location` → `loc_country` (e.g. "US"), `loc_state` (e.g. "CA")


In [7]:
def safe_json_get(val, key, default=None):
    """Extract key from JSON string with regex fallback."""
    try:
        if pd.isna(val):
            return default
        d = json.loads(str(val).replace("'", '"'))
        return d.get(key, default)
    except Exception:
        pattern = rf'"{re.escape(key)}"\s*:\s*"([^"]*)"'
        m = re.search(pattern, str(val))
        return m.group(1) if m else default

# Parse on full df (apply to both train and test consistently)
print("Parsing category column...")
df["cat_name"]        = df["category"].apply(lambda x: safe_json_get(x, "name"))
df["cat_parent_name"] = df["category"].apply(lambda x: safe_json_get(x, "parent_name"))

print("Parsing location column...")
df["loc_country"] = df["location"].apply(lambda x: safe_json_get(x, "country"))
df["loc_state"]   = df["location"].apply(lambda x: safe_json_get(x, "state"))

print(f"\ncat_name unique values     : {df['cat_name'].nunique()}")
print(f"cat_parent_name unique vals: {df['cat_parent_name'].nunique()}")
print(f"loc_country unique values  : {df['loc_country'].nunique()}")
print(f"\nSample parsed values:")
print(df[["cat_name", "cat_parent_name", "loc_country", "loc_state"]].head(5))


Parsing category column...
Parsing location column...

cat_name unique values     : 161
cat_parent_name unique vals: 15
loc_country unique values  : 207

Sample parsed values:
           cat_name cat_parent_name loc_country loc_state
0          Software      Technology        None      None
1        Journalism            None          US        NY
2           Puzzles           Games        None      None
3          Software      Technology          US        NY
4  Electronic Music           Music          US        LA


## Step 7 — Drop Leakage Columns

These columns must be **unconditionally removed** before any modelling.
They contain information that is only available **after** the campaign ends —
using them would cause catastrophic data leakage: the model would learn to
"cheat" by reading the answer from the data.

| Column | Why it's leakage |
|---|---|
| `pledged` | Total money raised — only known after campaign ends |
| `usd_pledged` | Same as pledged, in USD |
| `converted_pledged_amount` | Same, currency-converted |
| `backers_count` | Number of backers — only known after campaign ends |
| `percent_funded` | `pledged / goal × 100` — the answer expressed as a percentage |
| `spotlight` | **⚠️ MOST DANGEROUS**: Kickstarter awards spotlight status ONLY to successful campaigns. Pearson r = 1.0 with success. Including this would give 100% accuracy while learning nothing useful. |
| `state_changed_at` | Timestamp when outcome was recorded — post-campaign |
| `usd_exchange_rate` | Rate at settlement — post-campaign |
| `static_usd_rate` | Same |
| `fx_rate` | Same |


In [8]:
LEAKAGE_COLS = [
    "pledged", "usd_pledged", "converted_pledged_amount",
    "backers_count", "percent_funded", "spotlight",
    "state_changed_at", "usd_exchange_rate", "static_usd_rate", "fx_rate",
]

# Verify spotlight is perfectly correlated with success (Pearson r = 1.0)
if "spotlight" in df.columns:
    sp = df["spotlight"].map({"True": 1, "False": 0, True: 1, False: 0}).fillna(0)
    corr = sp.corr(df["success"])
    print(f"spotlight ↔ success Pearson r = {corr:.4f}  ← this MUST be dropped")

# Drop leakage columns that exist in the dataframe
cols_to_drop = [c for c in LEAKAGE_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"\nDropped {len(cols_to_drop)} leakage columns: {cols_to_drop}")
print(f"Shape after leakage drop: {df.shape}")

# Save reference list
with open(os.path.join(OUTPUTS_PATH, "leakage_columns.json"), "w") as f:
    json.dump({"leakage_columns": LEAKAGE_COLS}, f, indent=2)


spotlight ↔ success Pearson r = 1.0000  ← this MUST be dropped

Dropped 10 leakage columns: ['pledged', 'usd_pledged', 'converted_pledged_amount', 'backers_count', 'percent_funded', 'spotlight', 'state_changed_at', 'usd_exchange_rate', 'static_usd_rate', 'fx_rate']
Shape after leakage drop: (188429, 37)


## Step 8 — Drop Junk / Metadata Columns

These columns carry **no predictive signal** — they are URL links, image paths,
API metadata, or redundant identifiers. Keeping them would only add noise and
slow down training.


In [9]:
JUNK_COLS = [
    "photo", "profile", "urls", "creator", "source_url", "slug",
    "currency_symbol", "currency_trailing_code", "current_currency",
    "is_liked", "is_disliked", "is_starrable",
    "is_in_post_campaign_pledging_phase", "usd_type", "prelaunch_activated",
    # raw JSON columns (we've extracted what we need)
    "category", "location",
]

cols_to_drop_junk = [c for c in JUNK_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop_junk)
print(f"Dropped {len(cols_to_drop_junk)} junk columns")
print(f"Shape after junk drop: {df.shape}")
print(f"\nRemaining columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")


Dropped 17 junk columns
Shape after junk drop: (188429, 20)

Remaining columns (20):
  blurb
  country
  country_displayable_name
  created_at
  currency
  deadline
  disable_communication
  goal
  id
  is_launched
  launched_at
  name
  staff_pick
  state
  video
  success
  cat_name
  cat_parent_name
  loc_country
  loc_state


## Step 9 — Missing Value Audit

After cleaning, we audit every remaining column for missing values.
Missingness is **often informative**:
- Missing `video` → the creator did not include a video (a meaningful choice)
- Missing `cat_name` → the category JSON could not be parsed
- These are handled in feature engineering (e.g. `has_video = video.notna()`)


In [10]:
audit = pd.DataFrame({
    "dtype": df.dtypes,
    "n_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().mean() * 100).round(2),
    "n_unique": df.nunique(),
}).sort_values("pct_missing", ascending=False)

print("Missing value audit:")
print(audit.to_string())

# Notes on missingness
notes = {
    "video"           : "Missing = no video attached (informative, not random)",
    "cat_name"        : "Missing = JSON parse failure; very small fraction",
    "cat_parent_name" : "Same as cat_name",
    "loc_country"     : "Missing = location not set by creator",
    "loc_state"       : "Missing = location not set or non-US campaign",
    "blurb"           : "Missing = creator left description blank",
    "disable_communication": "Rare flag; treat as 0 if missing",
}
print("\nMissingness notes:")
for col, note in notes.items():
    if col in df.columns:
        n = df[col].isnull().sum()
        print(f"  [{col}]  ({n:,} missing): {note}")


Missing value audit:
                            dtype  n_missing  pct_missing  n_unique
video                      object      62016      32.9100    126413
cat_parent_name            object       4210       2.2300        15
loc_state                  object        195       0.1000      1196
loc_country                object        146       0.0800       207
country                    object          0       0.0000        25
cat_name                   object          0       0.0000       161
success                     int64          0       0.0000         2
state                      object          0       0.0000         2
staff_pick                   bool          0       0.0000         2
name                       object          0       0.0000    187869
blurb                      object          4       0.0000    185968
is_launched                  bool          0       0.0000         1
id                          int64          0       0.0000    188429
goal                      f

## Step 10 — Save Outputs

We save three parquet files and the leakage reference JSON.  
Parquet is used instead of CSV because it:
- Preserves column dtypes (no re-parsing needed)
- Is ~3–5× faster to read/write
- Compresses better for large DataFrames


In [11]:
# Re-apply JSON parsing to train/test splits (they're subsets of df before we dropped cols)
# Rebuild from the cleaned df using the original split indices
df["launched_at"] = pd.to_numeric(df["launched_at"], errors="coerce")
df = df.sort_values("launched_at").reset_index(drop=True)
cutoff_idx = int(len(df) * 0.80)
train_df = df.iloc[:cutoff_idx].copy()
test_df  = df.iloc[cutoff_idx:].copy()

# Save
clean_path = os.path.join(OUTPUTS_PATH, "clean_df.parquet")
train_path = os.path.join(OUTPUTS_PATH, "train_df.parquet")
test_path  = os.path.join(OUTPUTS_PATH, "test_df.parquet")

df.to_parquet(clean_path, index=False)
train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

print(f"Saved clean_df  : {len(df):,} rows  → {clean_path}")
print(f"Saved train_df  : {len(train_df):,} rows → {train_path}")
print(f"Saved test_df   : {len(test_df):,} rows  → {test_path}")
print(f"\nNotebook 01 complete. Proceed to 02_EDA.ipynb")


Saved clean_df  : 188,429 rows  → ../outputs/clean_df.parquet
Saved train_df  : 150,743 rows → ../outputs/train_df.parquet
Saved test_df   : 37,686 rows  → ../outputs/test_df.parquet

Notebook 01 complete. Proceed to 02_EDA.ipynb
